# Predicting Target Customers using Classification Decision Trees
In this notebook, we predict customers most likely to be positive responders to an Affinity Card loyalty program. 

High Affinity Card responders (target value = 1) are defined as those customers who, when given a loyalty or affinity card, hyper-respond, that is, increase purchases more than the Affinity Card program's offered discount. 

This notebook builds and applies a classification decision tree model using the Sales History (SH) schema data. All processing occurs inside Oracle Autonomous Database.


Copyright (c) 2025 Oracle Corporation 
###### <a href="https://oss.oracle.com/licenses/upl/" target="_blank">The Universal Permissive License (UPL), Version 1.0</a>

---

<dl>
<img src="https://www.oracle.com/technetwork/database/options/advanced-analytics/classification-5663162.jpg" alt="OML Notebooks" width="250"/>
</dl>

* <a href="https://docs.oracle.com/en/cloud/paas/autonomous-data-warehouse-cloud/index.html" target="_blank">Oracle ADW Documentation</a>
* <a href="https://github.com/oracle-samples/oracle-db-examples/tree/main/machine-learning" target="_blank">OML folder on Oracle GitHub</a>
* <a href="https://www.oracle.com/machine-learning" target="_blank">OML Web Page</a>
* <a href="https://www.oracle.com/goto/ml-classification" target="_blank">OML Classification</a>
* <a href="https://docs.oracle.com/en/database/oracle/machine-learning/oml4py/2/mlugp/decision-tree.html" target="_blank">OML4Py Decision Tree</a>

In [0]:
%pythonimport pandas as pd
import oml

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [0]:
%pythonDEMO = oml.sync(table = "SUPPLEMENTARY_DEMOGRAPHICS", schema = "SH")
z.show(DEMO.head())

In [0]:
%pythonDEMO.shape

In [0]:
%pythonz.show(DEMO.crosstab('AFFINITY_CARD'))

In [0]:
%pythonz.show(DEMO.crosstab(['HOUSEHOLD_SIZE', 'AFFINITY_CARD']))

In [0]:
%pythonDEMO_DF = DEMO[["CUST_ID", 'AFFINITY_CARD', "BOOKKEEPING_APPLICATION", "BULK_PACK_DISKETTES", "EDUCATION", 
 "FLAT_PANEL_MONITOR", "HOME_THEATER_PACKAGE", "HOUSEHOLD_SIZE", "OCCUPATION", "OS_DOC_SET_KANJI", 
 "PRINTER_SUPPLIES", "YRS_RESIDENCE", "Y_BOX_GAMES"]]

In [0]:
%pythonz.show(DEMO_DF.head())

In [0]:
%pythonTRAIN, TEST = DEMO_DF.split(ratio = (0.6,0.4))
TRAIN_X = TRAIN.drop('AFFINITY_CARD')
TRAIN_Y = TRAIN['AFFINITY_CARD']
TEST_X = TEST
TEST_Y = TEST['AFFINITY_CARD']

### Build a Decision Tree model to predict AFFINITY_CARD

---

In [0]:
%pythontry:
    oml.drop(model = 'DT_CLAS_MODEL') 
except:
    print("No such model")

setting = dict()
dt_mod = oml.dt(**setting)
dt_mod.fit(TRAIN_X, TRAIN_Y, case_id = 'CUST_ID', model_name = 'DT_CLAS_MODEL')

In [0]:
%pythondt_mod.score(TEST_X, TEST_Y)

### Examples of possible setting overrides for DT 

If the user does not override the default settings, then relevant settings are determined by the algorithm

A complete list of settings can be found in the Documentation link:

- Algorithm Settings: <a href="https://docs.oracle.com/en/database/oracle/machine-learning/oml4py/2/mlugp/decision-tree.html" target="_blank">Decision Tree</a> 

- Shared Settings: <a href="https://docs.oracle.com/en/database/oracle/machine-learning/oml4py/2/mlugp/shared-settings-23c.html" target="_blank">All algorithms</a>

* Specify a row weight column 
> 'ODMS_ROW_WEIGHT_COLUMN_NAME' : '<row_weight_column_name>'
* Specify a missing value treatment method.  The default is to replace with mean (numeric features), mode (categorical features) or delete the row
> 'ODMS_MISSING_VALUE_TREATMENT' : 'ODMS_MISSING_VALUE_DELETE_ROW'
* Specify Tree impurity metric for Decision Tree. Tree algorithms seek the best test question for splitting data at each node. The best splitter and split values are those that result in the largest increase in target value homogeneity (purity) for the entities in the node. Purity is by a metric. Decision trees can use either Gini `TREE_IMPURITY_GINI` or entropy `TREE_IMPURITY_ENTROPY` as the purity metric. By default, the algorithm uses `TREE_IMPURITY_GINI`.
> 'TREE_IMPURITY_METRIC' : 'TREE_IMPURITY_GINI'
* Specify the criteria for splits regarding the maximum tree depth (the maximum number of nodes between the root and any leaf node, including the leaf node). For Decision Tree, it requires a number between 2 and 20, and the default is 7. For Random Forest it is a number between 2 and 100, and the default is 16.
> 'TREE_TERM_MAX_DEPTH' : '7'
* Specify the minimum number of training rows in a node expressed as a percentage of the rows in the training data.  It requires a number between 0 and 10.  The default is 0.05, indicating 0.05%. 
> 'TREE_TERM_MINPCT_NODE' : '0.05'
* Specifyt he minimum number of rows required to consider splitting a node expressed as a percentage of the training rows.  It requires a number greater than 0, and smaller or equal to 20.  The default is 0.1, indicating 0.1%. 
> 'TREE_TERM_MINPCT_SPLIT' : '0.1'
* Specify The minimum number of rows in a node. It requires a number greater than or equal to zero. The default is 10.      
> 'TREE_TERM_MINREC_NODE' : '10'
* Specify the criteria for splits regarding the minimum number of records in a parent node expressed as a value.  No split is attempted if the number of records is below this value.  It requires a number greater than 1.  The default is 20.       
> 'TREE_TERM_MINREC_SPLIT' : '20'
* Specify the maximum number of bins for each attribute. For Decision Tree it requires a number between 2 and 2,147,483,647, with the default value of 32. For Random Forest it requires a number between 2 and 254, with the default value of 32.
> 'CLAS_MAX_SUP_BINS' : '32'

In [0]:
%pythontry:
    oml.drop(model = 'DT_CLAS_MODEL') 
except:
    print("No such model")

settings = {'TREE_IMPURITY_METRIC': 'TREE_IMPURITY_GINI', 
            'TREE_TERM_MAX_DEPTH': '7',
            'TREE_TERM_MINPCT_NODE': '0.05', 
            'TREE_TERM_MINPCT_SPLIT': '0.1',
            'TREE_TERM_MINREC_NODE': '10',
            'TREE_TERM_MINREC_SPLIT': '20',
            'CLAS_MAX_SUP_BINS': '32'} 


dt_mod = oml.dt(**settings)
dt_mod.fit(TRAIN_X, TRAIN_Y, case_id = 'CUST_ID', model_name = 'DT_CLAS_MODEL')

In [0]:
%pythondt_mod.score(TEST_X, TEST_Y)

In [0]:
%pythoncase_id = 'CUST_ID'
RES_DF = dt_mod.predict(TEST_X, supplemental_cols = TEST_X)
RES_PROB = dt_mod.predict_proba(TEST_X, supplemental_cols = TEST_X[case_id])

# Join the entire result into RES_DF
RES_DF = RES_DF.merge(RES_PROB, how = "inner", on = case_id, suffixes = ["", ""])

In [0]:
%pythondef evaluate_model(pred_data='', 
    settings_name={''}, 
    name='',
    target=''
    ):
    """Evaluate the models by passing an proxy oml.Dataframe containing Predictions
     and the target column,
     The Settings name (for the charts), 
     The name of the model used (for the charts),
     Supply the target column name for evaluation
     for computing the confusion matrix with the test dataset"""
    import oml
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import auc
    from sklearn.metrics import roc_curve

    conf_matrix = pred_data.crosstab(target,'PREDICTION',pivot=True)
 
    # Extract Statistics from the Confusion Matrix
    cf_local = conf_matrix.pull()
    TN = int(cf_local[cf_local[target]==0]['count_(0)'])
    FN = int(cf_local[cf_local[target]==0]['count_(1)'])
    TP = int(cf_local[cf_local[target]==1]['count_(1)'])
    FP = int(cf_local[cf_local[target]==1]['count_(0)'])
    TPR = TP/(TP+FN)
    FPR = FP/(FP+TN)
    TNR = TN/(TN+FP)
    FNR = FN/(FN+TP)
    Precision = TP/(TP+FP)
    Accuracy = (TP+TN)/(TP+TN+FP+FN)
    NPV = TN/(FN+TN)
    DetectionRate = TN/(FN+TN)
    BalancedAccuracy = (TPR+TNR)/2
    
    # Estimated AUC via Triangle (not very precise) could be
    # AUC = (1/2)*FPR*TPR + (1/2)*(1-FPR)*(1-TPR) + (1-FPR)*TPR
    # Compute real AUC using roc_curve by loading the
    # data locally and using the roc_curve() function
    pred_local = pred_data.pull()
    fpr, tpr, _ = roc_curve(pred_local[[target]],pred_local[['PROBABILITY_OF_1']])
    AUC = auc(fpr, tpr)
    opt_index = np.argmax(tpr - fpr)
    FPR_OPT = fpr[opt_index]
    TPR_OPT = tpr[opt_index]
    F1Score = 2*Precision*TPR/(Precision+TPR)
    MathewsCorrCoef = ((TP*TN)-(FP*FN))/((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))**0.5
    
    # Store all statistics to export
    statistics = {'Algorithm' : name,
                  'Algorithm_setting' : settings_name,
                  'TN' : TN,
                  'TP' : TP,
                  'FP' : FP,
                  'FN' : FN,
                  'TPR' : TPR,
                  'FPR' : FPR,
                  'TNR' : TNR,
                  'FNR' : FNR,
                  'Precision' : Precision,
                  'Accuracy' : Accuracy,
                  'NPV' : NPV,
                  'DetectionRate' : DetectionRate,
                  'BalancedAccuracy' : BalancedAccuracy,
                  'AUC' : AUC,
                  'F1Score' : F1Score,
                  'MathewsCorrCoef' : MathewsCorrCoef
                  }
    # Nice round stats for printing to screen
    TOTAL = TP+TN+FP+FN
    TN_P = round((TN/TOTAL*100),2)
    FP_P = round((FP/TOTAL*100),2)
    FN_P = round((FN/TOTAL*100),2)
    TP_P = round((TP/TOTAL*100),2)
    # Print the output nicely on Zeppelin native Table
    print("%table CONFUSION MATRIX\tPREDICTED 0\tPREDICTED 1\nACTUAL 0\t"+
          "True Negative: "+str(TN)+" ("+str(TN_P)+"%)\t"+
          "False Positive: "+str(FP)+" ("+str(FP_P)+"%)\nACTUAL 1\t"+
          "False Negative: "+str(FN)+" ("+str(FN_P)+"%)\t"+
          "True Positive: "+str(TP)+" ("+str(TP_P)+"%)\n"+
          "Accuracy: "+str(round(Accuracy*100,4))+"%\t"+
          "AUC: "+str(round(AUC,4))+"\t"+
          "F1Score: "+str(round(F1Score,4))
          )
    
    # Multiple Charts for Evaluation
    fig, axes = plt.subplots(nrows=1, ncols=4,figsize=[22,5])
    ax1, ax2, ax3, ax4 = axes.flatten()
    fig.suptitle('Evaluation of the '+str(name)+' Model, with settings: '+str(settings_name), size=16)
    
    # Statistics
    ax1.axis('off')
    
    # Function to return rounded numbers if the string is float, return
    # integers otherwise and return characters if not a number
    def round_if_float(content):
        try:
            val = float(content)
        except ValueError:
            return(content)
        else:
            if val.is_integer():
                return(int(content))
            else:
                return(round(float(content),4))

    for num, name in enumerate(statistics):
        ax1.text(0.01, 
        (-num*0.06+0.94),
        "{0}: {1}".format(name,round_if_float(statistics[name])),
        ha='left', 
        va='bottom', 
        fontsize=12)
    
    # Produce Lift Chart
    ax2.set_title('Lift Chart')
    data = pred_local.sort_values(by='PROBABILITY_OF_1', ascending=False)
    data['row_id'] = range(0,0+len(data))
    data['decile'] = ( data['row_id'] / (len(data)/10) ).astype(int)
    lift = data.groupby('decile')[target].agg(['count','sum'])
    lift.columns = ['count', target]
    lift['decile'] = range(1,11)
    
    data_ideal = pred_local.sort_values(by=target, ascending=False)
    data_ideal['row_id'] = range(0,0+len(data))
    data_ideal['decile'] = ( data_ideal['row_id'] / (len(data_ideal)/10) ).astype(int)
    lift_ideal = data_ideal.groupby('decile')[target].agg(['count','sum'])
    lift_ideal.columns = ['count', 'IDEAL']
    lift['IDEAL']=lift_ideal['IDEAL']
    
    ax2.bar(lift['decile'],lift['IDEAL']/lift['count'],
    color='darkorange', label='Ideal')
    ax2.bar(lift['decile'],lift[target]/lift['count'],
    color='blue', alpha=0.6, label='Model')
    ax2.axhline((lift[target]/lift['count']).mean(), 
    color='grey', linestyle='--', label='Avg TARGET')
    ax2.set_ylim(0,1.15)
    ax2.set_xlabel('Decile', size=13)
    ax2.set_ylabel('Percent of Actual Targets', size=13)
    # Print labels.
    for dec in lift['decile']:
        ax2.text(dec, lift[lift.decile==dec][target]/lift[lift.decile==dec]['count'] + 0.05, 
        ("%.0f" % int(round((lift[(lift.decile==dec)][target]/lift[lift.decile==dec]['count'])*100,0)))+"%",
        ha='center', va='bottom')
    ax2.legend(loc="upper right")
    
    # Produce Gains Chart
    ax3.set_title('Distributions of Predictions')
    pred_local[pred_local[target]==1]['PROBABILITY_OF_1'].rename("Target = 1").plot(kind='density', bw_method=0.1, grid=True, ax=ax3)
    pred_local[pred_local[target]==0]['PROBABILITY_OF_1'].rename("Target = 0").plot(kind='density', bw_method=0.1, grid=True, ax=ax3)
    ax3.axvline(.5, color='grey', linestyle='--', label='Cutoff at 0.5')
    ax3.set_xlim([0,1])
    ax3.set_xlabel('Probability of 1', size=13)
    ax3.set_ylabel('Density', size=13)
    ax3.legend(loc="upper right")
    
    # ROC curve Chart
    ax4.set_title('ROC Curve')
    ax4.plot(fpr, tpr, color='blue', lw=2, label='ROC curve')
    ax4.plot(FPR_OPT, TPR_OPT,  color='orange', markersize=6)
    ax4.plot([0, 1], [0, 1], lw=2, linestyle='--', color='grey', label='Random guess')
    ax4.annotate('Optimal Cutoff:\nTPR: '+str(round(TPR_OPT,2))+' FPR: '+str(round(FPR_OPT,2)),
                 fontsize=11, xy=(FPR_OPT, TPR_OPT), xycoords='data', xytext=(0.98, 0.54), 
                 textcoords='data', 
                 arrowprops=dict(facecolor='gray', shrink=0.1, width=2,
                                 connectionstyle='arc3, rad=0.3'), 
                 horizontalalignment='right', verticalalignment='top')
    ax4.annotate('AUC ='+str(round(AUC,4)), xy=(0.5, 0.35), 
                 xycoords='axes fraction', size=13)
    ax4.annotate('Precision ='+str(round(Precision,4)), xy=(0.45, 0.3), 
                 xycoords='axes fraction', size=13)
    ax4.annotate('Recall ='+str(round(TPR,4)), xy=(0.4, 0.25), 
                 xycoords='axes fraction', size=13)
    ax4.annotate('Accuracy ='+str(round(Accuracy,4)), xy=(0.35, 0.2),
                 xycoords='axes fraction', size=13)
    ax4.annotate('F1 Score ='+str(round(F1Score,4)), xy=(0.3, 0.15), 
                 xycoords='axes fraction', size=13)
    ax4.set_xlim([-0.02, 1.02])
    ax4.set_ylim([0.0, 1.02])
    ax4.set_xlabel('False Positive Rate', size=13)
    ax4.set_ylabel('True Positive Rate', size=13) 
    ax4.legend(loc="lower right")
    
    return(statistics, pred_local)
_ = evaluate_model(pred_data=RES_DF, settings_name='Gini,Max Depth:7,Min%Node:0.05,Min%Split:0.1', name='Decision Tree', target='AFFINITY_CARD')

In [0]:
%pythonz.show(RES_DF[RES_DF['PROBABILITY_OF_1'] > 0.5][['PREDICTION', 'PROBABILITY_OF_1'] + RES_DF.columns].head())

In [0]:
%pythonRES_DF = dt_mod.predict(TEST_X, supplemental_cols = TEST_X[['CUST_ID', 'AFFINITY_CARD']], topN_attrs = True)
z.show(RES_DF.head())

In [0]:
%pythontry:
    oml.drop(table = 'DT_TEST_TABLE') 
except:
    pass
_ = TEST_X.materialize(table = 'DT_TEST_TABLE')

## End of Script